# Predicting Stellar Class from Data

This notebook trains models to predict stellar class for the Predicting Stellar Class challenge, a kaggle competition using synthetic data based on observational data. The synthetic data are published on kaggle:

Yao Yan, Walter Reade, Elizabeth Park. Predicting Stellar Class. https://kaggle.com/competitions/playground-series-s6e6, 2026. Kaggle.

In [35]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('playground-series-s6e6')

print("Path to competition files:", path)

Path to competition files: /Users/jaquesgillis/.cache/kagglehub/competitions/playground-series-s6e6


In [36]:
from setup import * 
import os

os.listdir(path)

['test.csv', 'train.csv', 'sample_submission.csv']

In [49]:
df = pd.read_csv(path + '/train.csv')

In [50]:
df.head()

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
0,0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY
1,1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY
2,2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO
3,3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,GALAXY
4,4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,GALAXY


In [140]:
# Preprocess the data

X = df.copy()
X = X.drop(columns=['id'])

# class lists are the columns we will transform
# value lists will give us the names for the transformed columns
ordinal_class = ['class']
ordinal_values = X['class'].unique()
spectral_class = ['spectral_type']
galaxy_class = ['galaxy_population']
hot_spectral_values = X['spectral_type'].unique()
hot_galaxy_values = X['galaxy_population'].unique()

# ordinal encode the target, one hot encode the other two
one_hot = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ordinal = OrdinalEncoder(categories=[ordinal_values])

# one-hot encode
hot_spectral_columns = one_hot.fit_transform(X[spectral_class])
hot_galaxy_columns = one_hot.fit_transform(X[galaxy_class])
new_spectral_columns = pd.DataFrame(hot_spectral_columns)
new_spectral_columns.columns = hot_spectral_values
new_galaxy_columns = pd.DataFrame(hot_galaxy_columns)
new_galaxy_columns.columns = hot_galaxy_values

ordinal_column = ordinal.fit_transform(X[ordinal_class])
new_ordinal_column = pd.DataFrame(ordinal_column)
new_ordinal_column.columns = ['class']

X = pd.concat([X.drop(columns=ordinal_class+spectral_class+galaxy_class, axis=1), new_spectral_columns, new_galaxy_columns], axis=1)
y = ordinal_column

### Baseline model

Let's just train a random forest model on the features we have here as a baseline.

In [108]:
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (577347, 14)
y shape: (577347, 1)


In [109]:
# Using random forest with n=100, max depth = 9

rf_model = RandomForestClassifier(n_estimators=100, max_depth=9)
train_X, val_X, train_y, val_y = train_test_split(X, y)
rf_model.fit(train_X, train_y)
val_predictions = rf_model.predict(val_X)
print(f"Baseline balanced accuracy score: {balanced_accuracy_score(val_y, val_predictions)}")

/opt/miniconda3/envs/kaggle-astro/lib/python3.10/site-packages/sklearn/base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Baseline balanced accuracy score: 0.9155033483437185


### Comment

Our baseline is high, which is good and bad news. It's good news, because without doing any work, we can already predict stellar class with good accuracy. It's bad news because we have a small window in which to make improvements, and even making small improvements is likely to be hard.

In [54]:
X.head()

,alpha,delta,u,g,r,i,z,redshift,M,O/B,G/K,A/F,Red_Sequence,Blue_Cloud
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,0.0,0.0,1.0,0.0,0.0,1.0
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,0.0,0.0,1.0,0.0,0.0,1.0
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,0.0,0.0,0.0,1.0,1.0,0.0
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,0.0,0.0,1.0,0.0,0.0,1.0
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,0.0,0.0,1.0,0.0,0.0,1.0


In [141]:
# Let's try normalizing features

X_normalized = X.copy()

filter_types = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z']

for filter_type in filter_types:
    denominator = X_normalized[filter_type].mean()
    X_normalized[filter_type] = X_normalized[filter_type] / denominator
    
red_denominator = X_normalized['redshift'].mean()
X_normalized['redshift'] = X_normalized['redshift'] / red_denominator

In [142]:
X_normalized.head()

,alpha,delta,u,g,r,i,z,redshift,M,O/B,G/K,A/F,Red_Sequence,Blue_Cloud
0,0.813440,0.776714,1.135024,1.042285,1.019793,0.993715,0.977938,0.565568,0.0,0.0,1.0,0.0,0.0,1.0
1,0.704719,1.481439,0.925879,0.908593,0.880999,0.888908,0.881588,0.218460,0.0,0.0,1.0,0.0,0.0,1.0
2,0.989957,1.618750,0.937317,1.003420,1.060564,1.062115,1.079629,3.904898,0.0,0.0,0.0,1.0,1.0,0.0
3,1.243379,2.224419,1.038461,1.002069,0.952659,0.947714,0.940855,0.741353,0.0,0.0,1.0,0.0,0.0,1.0
4,0.780964,0.885879,0.967081,0.926902,0.913421,0.923656,0.925165,0.768544,0.0,0.0,1.0,0.0,0.0,1.0


In [110]:
# Try the same simple random forest model on normalized feature set
# Using random forest with n=100, max depth = 9

rf_model = RandomForestClassifier(n_estimators=100, max_depth=9)
train_X, val_X, train_y, val_y = train_test_split(X_normalized, y)
rf_model.fit(train_X, train_y)
val_predictions = rf_model.predict(val_X)
print(f"Balanced accuracy score on normalized data: {balanced_accuracy_score(val_y, val_predictions)}")

/opt/miniconda3/envs/kaggle-astro/lib/python3.10/site-packages/sklearn/base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Balanced accuracy score on normalized data: 0.914664204976397


In exploratory data analysis, we found definite interactions between the 'G/K' spectral type and filter type. So we'll combine those here.

In [156]:
# Combining G/K with filter type

X_combinations = X.copy()
X_min = X_normalized.copy()

filter_types = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z']
X_min = X[filter_types]
features = []

for filter_type in filter_types:
    X_combinations['G/K' + filter_type] = X_combinations['G/K'] * X_combinations[filter_type]
    features.append('G/K' + filter_type)


In [158]:
X_combinations.head()

,alpha,delta,u,g,r,i,z,redshift,M,O/B,...,A/F,Red_Sequence,Blue_Cloud,G/Kalpha,G/Kdelta,G/Ku,G/Kg,G/Kr,G/Ki,G/Kz
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,0.0,0.0,...,0.0,0.0,1.0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,0.0,0.0,...,0.0,0.0,1.0,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,0.0,0.0,...,1.0,1.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,0.0,0.0,...,0.0,0.0,1.0,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,0.0,0.0,...,0.0,0.0,1.0,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185


In [157]:
# Try the same simple random forest model on new feature set
# Using random forest with n=100, max depth = 9

rf_model = RandomForestClassifier(n_estimators=100, max_depth=9)
train_X, val_X, train_y, val_y = train_test_split(X_combinations, y)
rf_model.fit(train_X, train_y)
val_predictions = rf_model.predict(val_X)
print(f"Balanced accuracy score on combined features: {balanced_accuracy_score(val_y, val_predictions)}")

/opt/miniconda3/envs/kaggle-astro/lib/python3.10/site-packages/sklearn/base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Balanced accuracy score on combined features: 0.9165033530151897


### Comment

Adding a combined 'G/K' and filter type feature slightly raised the balanced accuracy score. However, based on the data we have now, we cannot say whether 'G' and 'K' (which are not present as distinct spectral types in the training data) would have the same interaction--or whether 'A', 'F', 'O', or 'B' as distinct spectral types would interact with filter type. So depending on what the complete data set looks like, this gain may vanish or may even turn into a loss because of increased sparsity in the processed data.

### Finding the right model(s)

First, let's just see how simple models do with this processed data.

In [159]:
# Try a different model

xgb = XGBClassifier()
train_X, val_X, train_y, val_y = train_test_split(X_combinations, y)
xgb.fit(train_X, train_y)
val_predictions = xgb.predict(val_X)
print(f"Balanced accuracy score with xgb and combined features: {balanced_accuracy_score(val_y, val_predictions)}")

Balanced accuracy score with xgb and combined features: 0.9526117520466316
